In [ ]:
import numpy
import seaborn 
import pandas as pd
import matplotlib.pyplot as plt 
import nltk
from nltk.corpus import stopwords
import re
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
train

In [ ]:
nltk.download('stopwords')

In [ ]:
stop_words = set(stopwords.words('english')) 

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http/S+', '', text)
    text = re.sub(r'@\w+','',text)
    text = re.sub(r'#\w+','',text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

In [ ]:
train['clean_text'] = train['text'].apply(clean_text)
test['clean_text'] = test['text'].apply(clean_text)

In [ ]:
def features(df):
    df['text_length'] = df['text'].str.len()
    df['word_count'] = df['text'].str.split().str.len()
    df['hashtag_count'] = df['text'].str.count('#')
    return df

In [ ]:
train = features(train)
test = features(test)

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,3))

In [ ]:
X_text = vectorizer.fit_transform(train['clean_text'])
X_test_kaggle = vectorizer.transform(test['clean_text'])

In [ ]:
num_f_train = train[['text_length', 'word_count', 'hashtag_count']].values
num_f_test = test[['text_length', 'word_count', 'hashtag_count']].values

In [ ]:
X = numpy.hstack((X_text.toarray(), num_f_train))
X_kaggle = numpy.hstack((X_test_kaggle.toarray(), num_f_test))

In [ ]:
y = train['target']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
model = XGBClassifier(
    learning_rate=0.01,
    n_estimators=200,
    max_depth=7,
    subsample=0.8,
    gamma=0.5,
    reg_alpha=0.1,
    reg_lambda=0.1,
    tree_method='hist',
    device='cuda'
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)
pred_kaggle = model.predict(X_kaggle)

In [ ]:
report = classification_report(y_test, pred)
print(report)

In [ ]:
submission = pd.DataFrame({'id': test['id'], 'target': pred_kaggle})
submission.to_csv('submission_nltk.csv', index=False)